# Importing Libraries

In [28]:
import numpy as np
from pynq import Overlay, allocate
import subprocess
import ctypes
import time
import csv
import gc
import os

# Parameters

In [29]:
# ---------------- Kyber parameters ----------------
KYBER_N = 256
KYBER_Q = 3329
ITERATIONS = 1000

# Variants to run: security level -> K (number of polynomials per polyvec)
VARIANTS = {512: 2, 768: 3, 1024: 4}
VARIANTS_TO_RUN = [512, 768, 1024]          # remove entries to run only some variants

# ---------------- Files ----------------
BITSTREAM_DIR = "bitstream_files"
BITSTREAM_PATTERN = BITSTREAM_DIR + "/polyvec_accel_{level}.bit"   # needs a matching .hwh next to it
SW_SOURCES = [
    "polyvec_sw/ntt.c", "polyvec_sw/poly.c", "polyvec_sw/polyvec.c",
    "polyvec_sw/reduce.c", "polyvec_sw/verify.c", "polyvec_sw/fips202.c" # fips202.c adicionado
]
SO_PATTERN = "kyber_ref_{level}.so"          # one .so per variant (K is a compile-time macro)

# ---------------- Platform ----------------
CPU_FREQ_HZ = 650_000_000
FPGA_FREQ_HZ = 50_000_000

# If True, a variant with validation errors is NOT benchmarked
SKIP_BENCH_ON_FAILURE = True

# ---------------- Op codes ----------------
OP_CODES = {
    0: "OP_POLYVEC_NTT",
    1: "OP_POLYVEC_INVNTT_TOMONT",
    2: "OP_POLYVEC_BASEMUL_ACC_MONTGOMERY",
    3: "OP_POLYVEC_ADD",
    4: "OP_POLYVEC_REDUCE",
    5: "OP_KECCAK_PERMUTE", # Nova operação adicionada
}

# Register names that the HLS IP must expose (checked right after loading each bitstream)
REQUIRED_POINTERS = ["mem_pv_r", "mem_pv_a", "mem_pv_b", "mem_p_r", "mem_keccak_state"]

# Compiling and Loading the C Reference (one library per variant)

In [30]:
ptr_int16 = ctypes.POINTER(ctypes.c_int16)
ptr_uint64 = ctypes.POINTER(ctypes.c_uint64) 

def compile_reference(level, k):
    so_path = SO_PATTERN.format(level=level)
    cmd = [
        "gcc", "-shared", "-fPIC",
        "-Wall", "-Wextra", "-Wpedantic", "-Wmissing-prototypes",
        "-Wredundant-decls", "-Wshadow", "-Wpointer-arith",
        "-O3", "-fomit-frame-pointer", "-z", "noexecstack",
        "-Wno-unused-result",
        f"-DKYBER_K={k}",
        *SW_SOURCES,
        "-o", so_path,
    ]
    try:
        subprocess.run(cmd, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        print(e.stderr)
        raise
    return so_path

def load_reference(level, k):
    so_path = compile_reference(level, k)
    lib = ctypes.CDLL(os.path.abspath(so_path))
    prefix = f"pqcrystals_kyber{level}_ref_"

    def fn(name, n_args):
        try:
            f = getattr(lib, prefix + name)
        except AttributeError:
            raise RuntimeError(f"Symbol {prefix}{name} not found in {so_path}.")
        f.argtypes = [ptr_int16] * n_args
        f.restype = None
        return f
        
    def fn_unprefixed(name, ptr_type):
        try:
            f = getattr(lib, name)
        except AttributeError:
            raise RuntimeError(f"Symbol {name} not found in {so_path}.")
        f.argtypes = [ptr_type]
        f.restype = None
        return f

    return {
        0: fn("polyvec_ntt", 1),
        1: fn("polyvec_invntt_tomont", 1),
        2: fn("polyvec_basemul_acc_montgomery", 3),
        3: fn("polyvec_add", 3),
        4: fn("polyvec_reduce", 1),
        5: fn_unprefixed("pqcrystals_kyber_fips202_ref_KeccakF1600_StatePermute", ptr_uint64),
    }

def sw_args(op, c_r_pv, c_r_p, c_a, c_b, c_keccak=None):
    if op in (0, 1, 4): return (c_r_pv,)
    if op == 2:         return (c_r_p, c_a, c_b)
    if op == 3:         return (c_r_pv, c_a, c_b)
    if op == 5:         return (c_keccak,)
    raise ValueError(op)

# Hardware Auxiliary Functions

In [31]:
def load_accelerator(level):
    bit_path = BITSTREAM_PATTERN.format(level=level)
    if not os.path.exists(bit_path):
        raise FileNotFoundError(bit_path)
    if not os.path.exists(bit_path.replace(".bit", ".hwh")):
        print(f"[warning] {bit_path.replace('.bit', '.hwh')} not found - PYNQ needs the .hwh to parse the IPs")

    overlay = Overlay(bit_path)

    # Find the accelerator without hard-coding the instance name
    ip_name = next((n for n in overlay.ip_dict if n.startswith("polyvec_accel")), None)
    if ip_name is None:
        raise RuntimeError(f"No polyvec_accel IP in {bit_path}. IPs found: {list(overlay.ip_dict)}")
    accel = getattr(overlay, ip_name)

    # Sanity check of the register map
    missing = [p for p in REQUIRED_POINTERS
               for suffix in ("_1", "_2") if not hasattr(accel.register_map, p + suffix)]
    if missing:
        raise RuntimeError(f"Registers missing in {bit_path}: {missing}")
    return overlay, accel


def set_pointer(accel, prefix_name, physical_address):
    lower_32 = physical_address & 0xFFFFFFFF
    upper_32 = (physical_address >> 32) & 0xFFFFFFFF
    setattr(accel.register_map, f"{prefix_name}_1", lower_32)
    setattr(accel.register_map, f"{prefix_name}_2", upper_32)


def run_hardware(accel, op, timeout_s=2.0):
    t0 = time.perf_counter()
    accel.write(0x10, op)  # write op_code
    accel.write(0x00, 1)   # start
    t1 = time.perf_counter()

    while (accel.read(0x00) & 0x02) == 0:  # wait for done
        if time.perf_counter() - t1 > timeout_s:
            raise TimeoutError(f"Accelerator did not finish op {op} (wrong bitstream/K?)")
    t2 = time.perf_counter()

    time_execution_us = (t2 - t1) * 1_000_000
    time_total_us = (t2 - t0) * 1_000_000
    return time_total_us, time_execution_us


def allocate_buffers(k):
    n = KYBER_N * k
    return {
        "pv_r": allocate(shape=(n,), dtype=np.int16, cacheable=True),
        "pv_a": allocate(shape=(n,), dtype=np.int16, cacheable=True),
        "pv_b": allocate(shape=(n,), dtype=np.int16, cacheable=True),
        "p_r":  allocate(shape=(KYBER_N,), dtype=np.int16, cacheable=True),
        "keccak_state": allocate(shape=(25,), dtype=np.uint64, cacheable=True)
    }

def free_buffers(bufs):
    for b in bufs.values():
        try:
            b.freebuffer()
        except Exception:
            pass

def map_pointers(accel, bufs):
    set_pointer(accel, "mem_pv_r", bufs["pv_r"].physical_address)
    set_pointer(accel, "mem_pv_a", bufs["pv_a"].physical_address)
    set_pointer(accel, "mem_pv_b", bufs["pv_b"].physical_address)
    set_pointer(accel, "mem_p_r",  bufs["p_r"].physical_address)
    set_pointer(accel, "mem_keccak_state", bufs["keccak_state"].physical_address)

# Validation Tests

In [32]:
def validate_variant(level, k, accel, bufs, sw_funcs):
    n = KYBER_N * k
    total_errors = 0
    rng = np.random.default_rng()

    for op_code, op_name in OP_CODES.items():
        print(f"[Kyber-{level}] Testing {op_name} ({ITERATIONS} iterations)")
        op_errors = 0

        for _ in range(ITERATIONS):
            if op_code == 5:
                state = rng.integers(0, 2**64, size=25, dtype=np.uint64)
                bufs["keccak_state"][:] = state
                ref_state = state.copy()
                
                bufs["keccak_state"].sync_to_device() 
                
                args = sw_args(op_code, None, None, None, None, ref_state.ctypes.data_as(ptr_uint64))
                sw_funcs[op_code](*args)
                run_hardware(accel, op_code)
                
                bufs["keccak_state"].sync_from_device() 
                
                hw_val, sw_val = bufs["keccak_state"], ref_state
                bad = np.where(hw_val != sw_val)[0]
                
            else:
                a_pv = np.random.randint(0, KYBER_Q, size=n, dtype=np.int16)
                b_pv = np.random.randint(0, KYBER_Q, size=n, dtype=np.int16)
                bufs["pv_a"][:] = a_pv
                bufs["pv_b"][:] = b_pv
                bufs["pv_r"][:] = 0
                bufs["p_r"][:] = 0

                ref_pv_r = np.zeros(n, dtype=np.int16)
                ref_p_r  = np.zeros(KYBER_N, dtype=np.int16)

                if op_code in (0, 1, 4):          
                    ref_pv_r[:] = a_pv
                    bufs["pv_r"][:] = a_pv

                # Sincroniza inputs polinomiais
                bufs["pv_a"].sync_to_device()
                bufs["pv_b"].sync_to_device()
                bufs["pv_r"].sync_to_device()
                bufs["p_r"].sync_to_device()

                args = sw_args(op_code,
                               ref_pv_r.ctypes.data_as(ptr_int16),
                               ref_p_r.ctypes.data_as(ptr_int16),
                               a_pv.ctypes.data_as(ptr_int16),
                               b_pv.ctypes.data_as(ptr_int16))
                sw_funcs[op_code](*args)
                run_hardware(accel, op_code)

                bufs["pv_r"].sync_from_device()
                bufs["p_r"].sync_from_device()

                if op_code == 2:
                    hw_val, sw_val = bufs["p_r"], ref_p_r
                else:
                    hw_val, sw_val = bufs["pv_r"], ref_pv_r

                diff = hw_val.astype(np.int32) - sw_val.astype(np.int32)
                bad = np.where(diff % KYBER_Q != 0)[0] 

            if len(bad) > 0:
                if op_errors == 0:
                    i0 = bad[0]
                    print(f"  Hardware mismatch starting at index {i0} ({len(bad)} bad elements)")
                    if op_code == 5:
                        print(f"  Input:     {hex(state[i0])}")
                        print(f"  HW output: {hex(hw_val[i0])}")
                        print(f"  SW output: {hex(sw_val[i0])}")
                    else:
                        print(f"  HW output: {hw_val[i0]}")
                        print(f"  SW output: {sw_val[i0]}")
                op_errors += 1
                total_errors += 1

        if op_errors == 0:
            print(f"  passed all {ITERATIONS} iterations.")
        else:
            print(f"  failed in {op_errors} out of {ITERATIONS} iterations.")
        print("-" * 50)

    return total_errors

# Benchmark

In [33]:
def benchmark_variant(level, k, accel, bufs, sw_funcs):
    n = KYBER_N * k
    rows = []
    rng = np.random.default_rng()

    c_a_pv = bufs["pv_a"].ctypes.data_as(ptr_int16)
    c_b_pv = bufs["pv_b"].ctypes.data_as(ptr_int16)
    c_r_pv = bufs["pv_r"].ctypes.data_as(ptr_int16)
    c_r_p  = bufs["p_r"].ctypes.data_as(ptr_int16)
    c_keccak = bufs["keccak_state"].ctypes.data_as(ptr_uint64)

    for op_code, op_name in OP_CODES.items():
        sw_fn = sw_funcs[op_code]
        args = sw_args(op_code, c_r_pv, c_r_p, c_a_pv, c_b_pv, c_keccak)

        total_time_sw = 0.0
        total_time_hw_total = 0.0
        total_time_hw_exec = 0.0
        
        bytes_input = (25 * 8) if op_code == 5 else (n * 2)

        for _ in range(ITERATIONS):
            if op_code == 5:
                bufs["keccak_state"][:] = rng.integers(0, 2**64, size=25, dtype=np.uint64)
            else:
                bufs["pv_a"][:] = np.random.randint(0, KYBER_Q, size=n, dtype=np.int16)
                bufs["pv_b"][:] = np.random.randint(0, KYBER_Q, size=n, dtype=np.int16)

            start_sw = time.perf_counter()
            sw_fn(*args)
            end_sw = time.perf_counter()
            total_time_sw += (end_sw - start_sw)

            hw_total_us, hw_exec_us = run_hardware(accel, op_code)
            total_time_hw_total += hw_total_us
            total_time_hw_exec  += hw_exec_us

        avg_sw_s       = total_time_sw / ITERATIONS
        avg_hw_total_s = (total_time_hw_total / ITERATIONS) / 1_000_000
        avg_hw_exec_s  = (total_time_hw_exec / ITERATIONS) / 1_000_000

        cycles_sw       = avg_sw_s * CPU_FREQ_HZ
        cycles_hw_total = avg_hw_total_s * FPGA_FREQ_HZ
        cycles_hw_exec  = avg_hw_exec_s * FPGA_FREQ_HZ

        throughput_sw       = 1.0 / avg_sw_s
        throughput_hw_total = 1.0 / avg_hw_total_s
        throughput_hw_exec  = 1.0 / avg_hw_exec_s

        cpb_sw       = cycles_sw / bytes_input
        cpb_hw_total = cycles_hw_total / bytes_input
        cpb_hw_exec  = cycles_hw_exec / bytes_input

        speedup_time_total = avg_sw_s / avg_hw_total_s if avg_hw_total_s > 0 else 0
        speedup_time_exec  = avg_sw_s / avg_hw_exec_s if avg_hw_exec_s > 0 else 0
        speedup_cycles_total = cycles_sw / cycles_hw_total if cycles_hw_total > 0 else 0
        speedup_cycles_exec  = cycles_sw / cycles_hw_exec if cycles_hw_exec > 0 else 0

        print(f"============ Kyber-{level} | {op_name} ============")
        print(f"{'Metric':<25} | {'CPU (SW)':<15} | {'FPGA (Exec)':<15} | {'FPGA (Total)':<15}")
        print("-" * 78)
        print(f"{'Average Time (us)':<25} | {avg_sw_s*1e6:<15.2f} | {avg_hw_exec_s*1e6:<15.2f} | {avg_hw_total_s*1e6:<15.2f}")
        print(f"{'Clock Cycles':<25} | {int(cycles_sw):<15} | {int(cycles_hw_exec):<15} | {int(cycles_hw_total):<15}")
        print(f"{'Throughput (ops/s)':<25} | {throughput_sw:<15.2f} | {throughput_hw_exec:<15.2f} | {throughput_hw_total:<15.2f}")
        print(f"{'Cycles per Byte (CPB)':<25} | {cpb_sw:<15.2f} | {cpb_hw_exec:<15.2f} | {cpb_hw_total:<15.2f}")
        print(f"{'Speedup (Time)':<25} | {'1.00x':<15} | {speedup_time_exec:<14.2f}x | {speedup_time_total:<14.2f}x")
        print(f"{'Speedup (Cycles)':<25} | {'1.00x':<15} | {speedup_cycles_exec:<14.2f}x | {speedup_cycles_total:<14.2f}x\n")

        rows.append({
            "Variant": f"Kyber{level}",
            "K": k,
            "Bytes_Input": bytes_input,
            "Operation": op_name,
            "SW_Time_us": round(avg_sw_s * 1e6, 2),
            "HW_Exec_Time_us": round(avg_hw_exec_s * 1e6, 2),
            "HW_Total_Time_us": round(avg_hw_total_s * 1e6, 2),
            "SW_Cycles": int(cycles_sw),
            "HW_Exec_Cycles": int(cycles_hw_exec),
            "HW_Total_Cycles": int(cycles_hw_total),
            "SW_Throughput_ops_s": round(throughput_sw, 2),
            "HW_Exec_Throughput_ops_s": round(throughput_hw_exec, 2),
            "HW_Total_Throughput_ops_s": round(throughput_hw_total, 2),
            "SW_CPB": round(cpb_sw, 2),
            "HW_Exec_CPB": round(cpb_hw_exec, 2),
            "HW_Total_CPB": round(cpb_hw_total, 2),
            "Speedup_Time_Exec": round(speedup_time_exec, 2),
            "Speedup_Time_Total": round(speedup_time_total, 2),
            "Speedup_Cycles_Exec": round(speedup_cycles_exec, 2),
            "Speedup_Cycles_Total": round(speedup_cycles_total, 2),
        })
    return rows

# Run All Variants (compile, load bitstream, validate, benchmark)

In [34]:
print(f"Frequencies -> CPU: {CPU_FREQ_HZ/1e6:.2f} MHz | FPGA: {FPGA_FREQ_HZ/1e6:.2f} MHz\n")

all_results = []
validation_summary = {}

for level in VARIANTS_TO_RUN:
    k = VARIANTS[level]
    print("#" * 78)
    print(f"# Kyber-{level} (K={k})")
    print("#" * 78)

    overlay = accel = bufs = None
    try:
        sw_funcs = load_reference(level, k)          # 1. C reference with the right K
        overlay, accel = load_accelerator(level)     # 2. bitstream of this variant
        bufs = allocate_buffers(k)                   # 3. buffers sized for K
        map_pointers(accel, bufs)                    # 4. physical addresses -> IP

        errors = validate_variant(level, k, accel, bufs, sw_funcs)
        validation_summary[level] = errors

        if errors > 0 and SKIP_BENCH_ON_FAILURE:
            print(f"Kyber-{level}: {errors} failed iterations -> benchmark skipped.\n")
            continue

        rows = benchmark_variant(level, k, accel, bufs, sw_funcs)
        all_results.extend(rows)

        # per-variant CSV
        fname = f"polyvec_benchmark_results_kyber{level}.csv"
        with open(fname, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader(); w.writerows(rows)
        print(f"Saved '{fname}'\n")

    except Exception as e:
        validation_summary[level] = f"ERROR: {e}"
        print(f"Kyber-{level} aborted: {e}\n")

    finally:
        # release CMA memory and drop the overlay before loading the next bitstream
        if bufs is not None:
            free_buffers(bufs)
        if overlay is not None and hasattr(overlay, "free"):
            try: overlay.free()
            except Exception: pass
        del overlay, accel, bufs
        gc.collect()

# combined CSV
if all_results:
    with open("polyvec_benchmark_results.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(all_results[0].keys()))
        w.writeheader(); w.writerows(all_results)
    print("Combined results saved to 'polyvec_benchmark_results.csv'")

print("\nValidation summary:")
for level, res in validation_summary.items():
    status = "OK" if res == 0 else (f"{res} failed iterations" if isinstance(res, int) else res)
    print(f"  Kyber-{level}: {status}")

Frequencies -> CPU: 650.00 MHz | FPGA: 50.00 MHz

##############################################################################
# Kyber-512 (K=2)
##############################################################################
[Kyber-512] Testing OP_POLYVEC_NTT (1000 iterations)
  passed all 1000 iterations.
--------------------------------------------------
[Kyber-512] Testing OP_POLYVEC_INVNTT_TOMONT (1000 iterations)
  passed all 1000 iterations.
--------------------------------------------------
[Kyber-512] Testing OP_POLYVEC_BASEMUL_ACC_MONTGOMERY (1000 iterations)
  passed all 1000 iterations.
--------------------------------------------------
[Kyber-512] Testing OP_POLYVEC_ADD (1000 iterations)
  passed all 1000 iterations.
--------------------------------------------------
[Kyber-512] Testing OP_POLYVEC_REDUCE (1000 iterations)
  passed all 1000 iterations.
--------------------------------------------------
[Kyber-512] Testing OP_KECCAK_PERMUTE (1000 iterations)
  passed all 100